<a href="https://colab.research.google.com/github/Kuo1204/114-1KUO-REPO-/blob/main/PPT2Course_AI_Step2_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!apt-get -qq update
!apt-get -qq install -y libreoffice poppler-utils ffmpeg
print('✅ 系統軟體安裝完成')


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Extracting templates from packages: 100%
Preconfiguring packages ...
Selecting previously unselected package fonts-opensymbol.
(Reading database ... 118243 files and directories currently installed.)
Preparing to unpack .../000-fonts-opensymbol_2%3a102.12+LibO7.3.7-0ubuntu0.22.04.12_all.deb ...
Unpacking fonts-opensymbol (2:102.12+LibO7.3.7-0ubuntu0.22.04.12) ...
Selecting previously unselected package libreoffice-style-colibre.
Preparing to unpack .../001-libreoffice-style-colibre_1%3a7.3.7-0ubuntu0.22.04.12_all.deb ...
Unpacking libreoffice-style-colibre (1:7.3.7-0ubuntu0.22.04.12) ...
Selecting previously unselected package libuno-sal3.
Preparing to unpack .../002-libuno-sal3_1%3a7.3.7-0ubuntu0.22.04.12_amd64.deb ...
Unpacking libuno-sal3 (1:7.3.7-0ubuntu0.22.04.12) ...
Selecting previously unsele

In [3]:
!pip -q install gradio python-pptx pdf2image python-docx reportlab edge-tts moviepy srt google-generativeai python-dotenv pypdf
print('✅ Python 套件安裝完成')


  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 26.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 23.9 MB/s eta 0:00:00
✅ Python 套件安裝完成


In [4]:
%%writefile utils.py
# -*- coding: utf-8 -*-
"""
utils.py
========
共用工具模組。

提供整個 PPT2Course AI 系統會用到的基礎功能：
1. 專案資料夾路徑管理 (PPT / Script / Audio / Images / Output / Assets)
2. 統一的 Logger（同時輸出到終端機 與 提供給 Gradio 即時 Log 使用的字串佇列）
3. 環境檢查（LibreOffice / FFmpeg 是否存在）
4. 常用小工具（清空資料夾、取得安全檔名等）

之後所有模組 (ppt_reader.py / script_generator.py / tts.py / subtitle.py / video.py / app.py)
都會 import 這個檔案。
"""

import os
import shutil
import logging
import subprocess
from datetime import datetime

# ------------------------------------------------------------
# 1. 專案根目錄與各功能資料夾路徑
# ------------------------------------------------------------
# BASE_DIR：本檔案所在目錄，作為整個專案的根目錄。
BASE_DIR = os.path.dirname(os.path.abspath(__file__))

# 定義六大工作資料夾。
DIR_PPT = os.path.join(BASE_DIR, "PPT")          # 使用者上傳的原始 PPT
DIR_SCRIPT = os.path.join(BASE_DIR, "Script")    # 產生/上傳的講稿 (docx, txt)
DIR_AUDIO = os.path.join(BASE_DIR, "Audio")      # AI 配音產生的 mp3/wav
DIR_IMAGES = os.path.join(BASE_DIR, "Images")    # PPT 每頁轉出的圖片
DIR_OUTPUT = os.path.join(BASE_DIR, "Output")    # 最終輸出 (mp4, srt, docx, pdf)
DIR_ASSETS = os.path.join(BASE_DIR, "Assets")    # 使用者上傳的 Logo / 背景音樂 / 片頭片尾

ALL_DIRS = [DIR_PPT, DIR_SCRIPT, DIR_AUDIO, DIR_IMAGES, DIR_OUTPUT, DIR_ASSETS]


def ensure_all_dirs():
    """
    確保六大工作資料夾都存在。
    若資料夾不存在則自動建立。
    在 app.py 啟動時應呼叫一次。
    """
    for d in ALL_DIRS:
        os.makedirs(d, exist_ok=True)


def make_session_dir(base_dir: str, session_id: str = None) -> str:
    """
    在指定的 base_dir 底下建立一個「本次工作階段」專屬的子資料夾。
    用途：避免多個使用者/多次上傳的檔案互相覆蓋。

    參數:
        base_dir: 例如 DIR_IMAGES 或 DIR_AUDIO
        session_id: 若未提供，會用目前時間戳記自動產生 (YYYYMMDD_HHMMSS)

    回傳:
        建立好的資料夾完整路徑
    """
    if session_id is None:
        session_id = datetime.now().strftime("%Y%m%d_%H%M%S")
    session_path = os.path.join(base_dir, session_id)
    os.makedirs(session_path, exist_ok=True)
    return session_path


def clear_dir(dir_path: str):
    """
    清空指定資料夾內的所有檔案與子資料夾（但保留該資料夾本身）。
    用於每次重新產生課程前，避免舊檔案殘留造成混淆。
    """
    if not os.path.exists(dir_path):
        os.makedirs(dir_path, exist_ok=True)
        return
    for name in os.listdir(dir_path):
        full_path = os.path.join(dir_path, name)
        if os.path.isdir(full_path):
            shutil.rmtree(full_path)
        else:
            os.remove(full_path)


def safe_filename(filename: str) -> str:
    """
    將檔名中可能造成路徑問題的字元移除，回傳安全的檔名。
    例如使用者上傳的檔名包含空白、特殊符號時使用。
    """
    keep_chars = (" ", ".", "_", "-")
    return "".join(c for c in filename if c.isalnum() or c in keep_chars).strip()


# ------------------------------------------------------------
# 2. Logger 設定
# ------------------------------------------------------------
# 建立一個全域 logger，同時：
#   (a) 印到終端機，方便本機/Colab 開發時除錯
#   (b) 寫入 LOG_MESSAGES 這個 list，讓 Gradio 介面可以輪詢並顯示「即時 Log」
LOG_MESSAGES = []  # Gradio 介面會定期讀取這個 list 來更新畫面上的 Log 區塊

logger = logging.getLogger("ppt2course")
logger.setLevel(logging.INFO)

if not logger.handlers:
    _console_handler = logging.StreamHandler()
    _console_handler.setFormatter(
        logging.Formatter("[%(asctime)s] [%(levelname)s] %(message)s", "%H:%M:%S")
    )
    logger.addHandler(_console_handler)


def log(message: str, level: str = "info"):
    """
    統一的紀錄函式。所有模組都應該呼叫這個函式，而不是直接 print()。

    參數:
        message: 要記錄的訊息文字
        level: "info" / "warning" / "error"

    效果:
        1. 依照 level 呼叫對應的 logging 函式（會顯示在終端機）
        2. 將帶時間戳記的訊息加進 LOG_MESSAGES，供 Gradio 前端顯示
    """
    timestamp = datetime.now().strftime("%H:%M:%S")
    line = f"[{timestamp}] {message}"
    LOG_MESSAGES.append(line)

    if level == "warning":
        logger.warning(message)
    elif level == "error":
        logger.error(message)
    else:
        logger.info(message)


def get_log_text() -> str:
    """
    回傳目前累積的所有 Log 訊息，串成單一字串（換行分隔）。
    給 Gradio 的 Textbox / Markdown 元件顯示用。
    """
    return "\n".join(LOG_MESSAGES)


def clear_log():
    """清空目前的 Log 紀錄，通常在每次開始新的一次課程生成時呼叫。"""
    LOG_MESSAGES.clear()


# ------------------------------------------------------------
# 3. 環境檢查（LibreOffice / FFmpeg）
# ------------------------------------------------------------
def check_command_exists(command: str) -> bool:
    """
    檢查系統上是否安裝了某個命令列工具（例如 soffice、ffmpeg）。
    使用 shutil.which，跨平台 (Windows / Linux / Colab) 皆可運作。
    """
    return shutil.which(command) is not None


def check_environment() -> dict:
    """
    檢查系統執行 PPT2Course AI 所需的外部工具是否齊全。

    回傳一個 dict，例如：
    {
        "libreoffice": True,
        "ffmpeg": True,
        "all_ok": True
    }

    使用時機：
        app.py 啟動時呼叫一次，若缺少工具則在畫面上提示使用者，
        避免執行到一半才失敗，讓使用者不知道原因。
    """
    # Windows 上 LibreOffice 命令通常是 soffice.exe，但 shutil.which 對兩者都適用。
    has_libreoffice = check_command_exists("soffice") or check_command_exists("libreoffice")
    has_ffmpeg = check_command_exists("ffmpeg")

    result = {
        "libreoffice": has_libreoffice,
        "ffmpeg": has_ffmpeg,
        "all_ok": has_libreoffice and has_ffmpeg,
    }

    if not has_libreoffice:
        log("⚠️ 找不到 LibreOffice (soffice)，PPT 轉圖片功能將無法使用。", "warning")
    if not has_ffmpeg:
        log("⚠️ 找不到 FFmpeg，影片合成功能將無法使用。", "warning")

    return result


def run_subprocess(cmd_list: list, description: str = "") -> subprocess.CompletedProcess:
    """
    統一的外部命令執行函式（例如呼叫 soffice / ffmpeg）。
    會自動記錄 log，並在失敗時記錄錯誤內容，方便除錯。

    參數:
        cmd_list: 命令列參數列表，例如 ["soffice", "--headless", "--convert-to", "pdf", "a.pptx"]
        description: 給 log 用的說明文字，例如 "轉換 PPT 為 PDF"

    回傳:
        subprocess.CompletedProcess 物件
    """
    if description:
        log(f"執行中：{description}")

    result = subprocess.run(
        cmd_list,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True,
    )

    if result.returncode != 0:
        log(f"❌ 命令執行失敗：{' '.join(cmd_list)}\n{result.stderr}", "error")
    else:
        log(f"✅ 完成：{description or ' '.join(cmd_list)}")

    return result


if __name__ == "__main__":
    # 簡單自我測試：確保資料夾建立、Log、環境檢查都正常運作。
    ensure_all_dirs()
    log("utils.py 自我測試開始")
    env = check_environment()
    print("環境檢查結果：", env)
    print("目前 Log：")
    print(get_log_text())


Writing utils.py


In [5]:
%%writefile ppt_reader.py
# -*- coding: utf-8 -*-
"""
ppt_reader.py
=============
負責所有「讀取 PowerPoint 檔案」相關的工作：

1. 使用 python-pptx 讀取每一頁投影片的：
   - 文字內容 (標題、內文)
   - 講者備忘稿 (Speaker Notes)
2. 使用 LibreOffice 將 PPT 轉成 PDF，
   再用 pdf2image 把 PDF 每一頁轉成 PNG 圖片
   （這些圖片會被後續 video.py 用來合成影片畫面）

之後 script_generator.py 會使用這裡回傳的文字資料來生成講稿，
video.py 會使用這裡輸出的圖片來合成影片。
"""

import os
from pptx import Presentation
from pdf2image import convert_from_path

from utils import log, run_subprocess, check_command_exists


# ------------------------------------------------------------
# 1. 讀取投影片文字內容與備忘稿
# ------------------------------------------------------------
def read_ppt_slides(ppt_path: str) -> list:
    """
    讀取 PPT 檔案，回傳每一頁的文字資訊。

    參數:
        ppt_path: PPT 檔案路徑 (.pptx)

    回傳:
        一個 list，每個元素是一個 dict，代表一頁投影片，例如：
        [
            {
                "slide_index": 1,           # 第幾頁 (從1開始)
                "title": "課程介紹",         # 該頁的標題文字（若有偵測到）
                "body_text": "本課程將...",  # 該頁內文字（不含標題），用換行合併
                "notes": "各位同仁大家好..."  # 該頁的 Speaker Notes（若無則為空字串）
            },
            ...
        ]

    設計理由:
        將標題與內文分開，是因為 AI 生成講稿時（模式二）
        通常需要知道「這頁的主題是什麼」，標題是很好的線索。
    """
    if not os.path.exists(ppt_path):
        raise FileNotFoundError(f"找不到 PPT 檔案: {ppt_path}")

    log(f"開始讀取 PPT 內容：{os.path.basename(ppt_path)}")
    prs = Presentation(ppt_path)

    slides_data = []

    for idx, slide in enumerate(prs.slides, start=1):
        title_text = ""
        body_lines = []

        for shape in slide.shapes:
            if not shape.has_text_frame:
                continue

            text = shape.text_frame.text.strip()
            if not text:
                continue

            # 判斷是否為標題：
            # python-pptx 的 placeholder type 10 (TITLE) 或 13 (CENTER_TITLE) 代表標題文字框。
            is_title_placeholder = False
            if shape.is_placeholder:
                try:
                    ph_type = shape.placeholder_format.type
                    # 用 str 判斷避免不同 python-pptx 版本 enum 對應不到
                    if ph_type is not None and "TITLE" in str(ph_type):
                        is_title_placeholder = True
                except Exception:
                    pass

            if is_title_placeholder and not title_text:
                title_text = text
            else:
                body_lines.append(text)

        # 讀取 Speaker Notes（講者備忘稿）
        notes_text = ""
        if slide.has_notes_slide:
            notes_slide = slide.notes_slide
            if notes_slide.notes_text_frame is not None:
                notes_text = notes_slide.notes_text_frame.text.strip()

        slide_info = {
            "slide_index": idx,
            "title": title_text,
            "body_text": "\n".join(body_lines),
            "notes": notes_text,
        }
        slides_data.append(slide_info)

    log(f"✅ 共讀取到 {len(slides_data)} 頁投影片內容")
    return slides_data


# ------------------------------------------------------------
# 2. 將 PPT 轉換成每頁圖片 (PPT -> PDF -> PNG)
# ------------------------------------------------------------
def convert_ppt_to_images(ppt_path: str, output_dir: str, dpi: int = 150) -> list:
    """
    將 PPT 檔案的每一頁轉換成 PNG 圖片，供影片合成使用。

    流程:
        1. 呼叫 LibreOffice (soffice --headless) 把 .pptx 轉成 .pdf
        2. 用 pdf2image 把 PDF 逐頁轉成 PNG 圖片，存到 output_dir

    參數:
        ppt_path: PPT 檔案路徑
        output_dir: 圖片輸出資料夾
        dpi: 圖片解析度，數字越大圖片越清晰但檔案越大（預設150已足夠1080p影片使用）

    回傳:
        圖片路徑的 list，依照頁碼順序排列，例如：
        ["Images/xxx/slide_01.png", "Images/xxx/slide_02.png", ...]
    """
    if not check_command_exists("soffice") and not check_command_exists("libreoffice"):
        raise EnvironmentError(
            "找不到 LibreOffice，無法將 PPT 轉換成圖片。"
            "請先安裝 LibreOffice（Windows/Colab/HF Spaces 安裝方式請參考 README）。"
        )

    os.makedirs(output_dir, exist_ok=True)

    # --- Step 1: PPT -> PDF ---
    soffice_cmd = "soffice" if check_command_exists("soffice") else "libreoffice"
    log(f"轉換 PPT 為 PDF 中（使用 {soffice_cmd}）...")

    run_subprocess(
        [
            soffice_cmd,
            "--headless",
            "--convert-to",
            "pdf",
            "--outdir",
            output_dir,
            ppt_path,
        ],
        description="PPT 轉換為 PDF",
    )

    ppt_basename = os.path.splitext(os.path.basename(ppt_path))[0]
    pdf_path = os.path.join(output_dir, f"{ppt_basename}.pdf")

    if not os.path.exists(pdf_path):
        raise RuntimeError(f"PPT 轉 PDF 失敗，找不到輸出檔案：{pdf_path}")

    # --- Step 2: PDF -> PNG (逐頁) ---
    log("將 PDF 逐頁轉換為圖片中...")
    pages = convert_from_path(pdf_path, dpi=dpi)

    image_paths = []
    for i, page in enumerate(pages, start=1):
        image_path = os.path.join(output_dir, f"slide_{i:03d}.png")
        page.save(image_path, "PNG")
        image_paths.append(image_path)

    log(f"✅ 共產生 {len(image_paths)} 張投影片圖片，存放於 {output_dir}")
    return image_paths


# ------------------------------------------------------------
# 3. 整合函式：一次讀取文字 + 產生圖片
# ------------------------------------------------------------
def load_presentation(ppt_path: str, image_output_dir: str, dpi: int = 150) -> dict:
    """
    整合函式：同時讀取 PPT 文字內容，並轉換出每頁圖片。
    這是提供給 app.py 呼叫的主要入口函式。

    回傳:
        {
            "slides_text": [...],     # read_ppt_slides() 的結果
            "slides_images": [...],   # convert_ppt_to_images() 的結果
            "slide_count": 10
        }
    """
    slides_text = read_ppt_slides(ppt_path)
    slides_images = convert_ppt_to_images(ppt_path, image_output_dir, dpi=dpi)

    if len(slides_text) != len(slides_images):
        log(
            f"⚠️ 注意：文字頁數({len(slides_text)}) 與 圖片頁數({len(slides_images)}) 不一致，"
            "請確認 PPT 檔案是否正常。",
            "warning",
        )

    return {
        "slides_text": slides_text,
        "slides_images": slides_images,
        "slide_count": len(slides_text),
    }


# ------------------------------------------------------------
# 自我測試區塊
# ------------------------------------------------------------
if __name__ == "__main__":
    """
    直接執行本檔案時，會：
    1. 建立一份範例 PPT（3頁，含標題/內文/備忘稿）
    2. 測試文字讀取功能
    3. 測試轉圖片功能
    用來驗證整個模組在目前環境下可以正常運作。
    """
    from pptx.util import Inches

    test_ppt_path = "/home/claude/PPT2Course_AI/PPT/_test_sample.pptx"
    test_image_dir = "/home/claude/PPT2Course_AI/Images/_test_sample"

    # 建立測試用 PPT
    prs = Presentation()
    layout = prs.slide_layouts[1]  # 標題 + 內文版面

    sample_slides = [
        ("課程介紹", "本課程將說明公司資訊安全政策的重要性。", "各位同仁大家好，歡迎參加本次教育訓練課程。"),
        ("資安法規概述", "個人資料保護法\n資通安全管理法", ""),
        ("結語與Q&A", "感謝大家的參與，如有問題歡迎提出。", "課程到此結束，謝謝大家。"),
    ]

    for title, body, notes in sample_slides:
        slide = prs.slides.add_slide(layout)
        slide.shapes.title.text = title
        slide.placeholders[1].text = body
        if notes:
            slide.notes_slide.notes_text_frame.text = notes

    prs.save(test_ppt_path)
    print(f"已建立測試 PPT：{test_ppt_path}")

    # 測試整合函式
    result = load_presentation(test_ppt_path, test_image_dir)

    print("\n===== 文字讀取結果 =====")
    for s in result["slides_text"]:
        print(s)

    print("\n===== 圖片產生結果 =====")
    for p in result["slides_images"]:
        print(p, "存在:", os.path.exists(p))


Writing ppt_reader.py


In [6]:
%%writefile script_generator.py
# -*- coding: utf-8 -*-
"""
script_generator.py
====================
負責「講稿生成」的四種模式：

模式一 (own)      ：使用者提供 docx/txt 講稿，依頁碼標記自動對應每一頁投影片
模式二 (ai_auto)  ：AI 根據投影片文字，自動生成講稿 (使用 Gemini API)
模式三 (notes)    ：直接使用 PPT 內建的 Speaker Notes 當作講稿
模式四 (ai_polish)：AI 修飾使用者提供的講稿，讓語氣更口語自然

所有模式最終都會回傳「長度 = 投影片頁數」的講稿清單 (list[str])，
方便後續 tts.py 依序拿去產生配音。
"""

import os
import re
from docx import Document

from utils import log


# ------------------------------------------------------------
# 參考資料讀取 (讓 AI 依據真實資料生成/修飾講稿，避免內容失真)
# ------------------------------------------------------------
def extract_text_from_pdf(pdf_path: str) -> str:
    """讀取 PDF 檔案的純文字內容（逐頁擷取後合併）。"""
    from pypdf import PdfReader

    reader = PdfReader(pdf_path)
    texts = []
    for page in reader.pages:
        texts.append(page.extract_text() or "")
    return "\n".join(texts)


def extract_text_from_reference_docx(docx_path: str) -> str:
    """讀取 docx 參考資料的純文字內容。"""
    doc = Document(docx_path)
    return "\n".join(p.text for p in doc.paragraphs)


def extract_text_from_reference_txt(txt_path: str) -> str:
    """讀取 txt 參考資料的純文字內容。"""
    with open(txt_path, "r", encoding="utf-8", errors="ignore") as f:
        return f.read()


def extract_reference_text(file_paths: list, max_chars_per_file: int = 20000) -> str:
    """
    讀取一份或多份「參考資料」檔案 (PDF / docx / txt)，合併成一段文字，
    供 AI 生成或修飾講稿時，當作「真實資料依據」放進 prompt 一起參考。

    參數:
        file_paths: 參考資料檔案路徑清單
        max_chars_per_file: 每份檔案最多擷取的字數，避免單一檔案過長
                            導致整段 prompt 太大（Gemini 雖然容許很長的內容，
                            但過長仍會拖慢速度、增加費用），超過會自動截斷。

    回傳:
        合併後的參考資料文字（每份資料前會標註來源檔名），
        若沒有提供任何檔案，回傳空字串。
    """
    if not file_paths:
        return ""

    blocks = []
    for path in file_paths:
        if not path:
            continue
        ext = os.path.splitext(path)[1].lower()
        name = os.path.basename(path)

        try:
            if ext == ".pdf":
                text = extract_text_from_pdf(path)
            elif ext == ".docx":
                text = extract_text_from_reference_docx(path)
            elif ext == ".txt":
                text = extract_text_from_reference_txt(path)
            else:
                log(f"⚠️ 不支援的參考資料格式，已略過：{name}", "warning")
                continue
        except Exception as e:
            log(f"⚠️ 讀取參考資料失敗，已略過：{name}（原因：{e}）", "warning")
            continue

        text = text.strip()
        if not text:
            log(f"⚠️ 參考資料內容是空的，已略過：{name}", "warning")
            continue

        if len(text) > max_chars_per_file:
            text = text[:max_chars_per_file] + "\n...(內容過長，已自動截斷)"

        blocks.append(f"【參考資料來源：{name}】\n{text}")
        log(f"✅ 已讀取參考資料：{name}（擷取 {len(text)} 字）")

    return "\n\n".join(blocks)


# ------------------------------------------------------------
# 共用：頁碼標記格式
# ------------------------------------------------------------
# 支援的頁碼標記寫法，例如：
#   第1頁 / 第 1 頁 / 第1頁：
#   Slide1 / Slide 1
#   P1
#   投影片1 / 投影片 1
PAGE_MARKER_PATTERN = re.compile(
    r"^(?:第\s*(\d+)\s*頁|[Ss]lide\s*(\d+)|P(\d+)|投影片\s*(\d+))\s*[:：]?\s*$"
)


def format_scripts_for_display(slides_text: list, scripts: list) -> str:
    """
    將「每頁講稿清單」轉換成一份好閱讀、也方便使用者直接編輯的文字內容。
    格式範例：
        ===== 第1頁：課程介紹 =====
        各位同仁大家好...

        ===== 第2頁：資安法規概述 =====
        (講稿內容)

    這段文字會顯示在 Gradio 的編輯框中，使用者可以直接修改文字內容，
    改完後系統會用 parse_scripts_from_display() 重新解析回清單。
    """
    blocks = []
    for s, script in zip(slides_text, scripts):
        header = f"===== 第{s['slide_index']}頁：{s['title'] or '(無標題)'} ====="
        content = script.strip() if script else "(尚無內容，請手動輸入或重新產生)"
        blocks.append(f"{header}\n{content}")
    return "\n\n".join(blocks)


def parse_scripts_from_display(display_text: str, slide_count: int) -> list:
    """
    將使用者在編輯框中修改過的文字，重新解析回「每頁一則」的講稿清單。
    解析規則：以 "===== 第X頁" 開頭的那一行作為分頁標記。
    """
    lines = display_text.split("\n")
    scripts = ["" for _ in range(slide_count)]

    header_pattern = re.compile(r"^=+\s*第(\d+)頁")
    marker_indices = []
    for i, line in enumerate(lines):
        m = header_pattern.match(line.strip())
        if m:
            marker_indices.append((int(m.group(1)), i))

    for idx, (page_num, line_idx) in enumerate(marker_indices):
        start = line_idx + 1
        end = marker_indices[idx + 1][1] if idx + 1 < len(marker_indices) else len(lines)
        content = "\n".join(lines[start:end]).strip()
        if content == "(尚無內容，請手動輸入或重新產生)":
            content = ""
        if 1 <= page_num <= slide_count:
            scripts[page_num - 1] = content

    return scripts


# ------------------------------------------------------------
# 模式一：讀取使用者提供的 docx / txt 講稿
# ------------------------------------------------------------
def _split_lines_by_slide(lines: list, slide_count: int) -> list:
    """
    共用邏輯：把一堆文字行，依照頁碼分割成對應投影片頁數的清單。

    優先順序：
        1. 若偵測到「第X頁」等頁碼標記 -> 依標記精準分割
        2. 若沒有標記 -> 改用「空白行分段」，依序對應第1、2、3...頁
           (適合使用者只是單純每頁空一行寫講稿的情況)
    """
    marker_indices = []
    for i, line in enumerate(lines):
        m = PAGE_MARKER_PATTERN.match(line.strip())
        if m:
            num = next(g for g in m.groups() if g is not None)
            marker_indices.append((int(num), i))

    scripts = ["" for _ in range(slide_count)]

    if marker_indices:
        log(f"偵測到 {len(marker_indices)} 個頁碼標記，依標記分割講稿")
        for idx, (page_num, line_idx) in enumerate(marker_indices):
            start = line_idx + 1
            end = marker_indices[idx + 1][1] if idx + 1 < len(marker_indices) else len(lines)
            content = "\n".join(l for l in lines[start:end] if l.strip()).strip()
            if 1 <= page_num <= slide_count:
                scripts[page_num - 1] = content
            else:
                log(f"⚠️ 講稿中標記的頁碼 {page_num} 超出投影片範圍(共{slide_count}頁)，已略過", "warning")
        return scripts

    log("未偵測到頁碼標記，改用「空白行分段」方式依序對應投影片頁數", "warning")
    blocks = []
    current = []
    for l in lines:
        if l.strip() == "":
            if current:
                blocks.append("\n".join(current).strip())
                current = []
        else:
            current.append(l)
    if current:
        blocks.append("\n".join(current).strip())

    for i in range(slide_count):
        if i < len(blocks):
            scripts[i] = blocks[i]

    if len(blocks) != slide_count:
        log(
            f"⚠️ 講稿段落數({len(blocks)}) 與投影片頁數({slide_count}) 不一致，"
            "建議在講稿中加入「第1頁」「第2頁」等標記，確保對應正確。",
            "warning",
        )

    return scripts


def read_script_from_docx(docx_path: str, slide_count: int) -> list:
    """讀取 .docx 講稿檔案，回傳對應每頁投影片的講稿清單。"""
    log(f"讀取使用者講稿 (Word)：{os.path.basename(docx_path)}")
    doc = Document(docx_path)
    lines = [p.text for p in doc.paragraphs]
    return _split_lines_by_slide(lines, slide_count)


def read_script_from_txt(txt_path: str, slide_count: int) -> list:
    """讀取 .txt 講稿檔案，回傳對應每頁投影片的講稿清單。"""
    log(f"讀取使用者講稿 (txt)：{os.path.basename(txt_path)}")
    with open(txt_path, "r", encoding="utf-8") as f:
        lines = f.read().split("\n")
    return _split_lines_by_slide(lines, slide_count)


# ------------------------------------------------------------
# 模式三：直接使用 PPT Speaker Notes
# ------------------------------------------------------------
def get_notes_scripts(slides_text: list) -> list:
    """
    模式三：直接把每頁的 Speaker Notes 當作講稿使用。
    slides_text 來自 ppt_reader.read_ppt_slides() 的回傳結果。
    """
    scripts = []
    empty_count = 0
    for s in slides_text:
        if not s["notes"]:
            empty_count += 1
        scripts.append(s["notes"])

    if empty_count > 0:
        log(f"⚠️ 有 {empty_count} 頁投影片沒有備忘稿內容，該頁講稿會是空白", "warning")
    log("✅ 已讀取所有投影片的 Speaker Notes 作為講稿")
    return scripts


# ------------------------------------------------------------
# 模式二 & 模式四：呼叫 Gemini API
# ------------------------------------------------------------
def _call_gemini(prompt: str, api_key: str, model_name: str = "gemini-1.5-flash") -> str:
    """
    共用函式：呼叫 Gemini API 並回傳純文字結果。
    之後若要切換其他 LLM (例如 OpenAI / Claude)，只需要修改這個函式即可，
    不影響上層的四種模式邏輯。
    """
    import google.generativeai as genai

    genai.configure(api_key=api_key)
    model = genai.GenerativeModel(model_name)
    response = model.generate_content(prompt)
    return (response.text or "").strip()


def generate_script_with_ai(
    slide_info: dict,
    api_key: str,
    reference_text: str = "",
    model_name: str = "gemini-1.5-flash",
) -> str:
    """
    模式二：根據單一頁投影片的文字內容，請 AI 生成一段自然口語的旁白講稿。

    參數:
        reference_text: 選填。使用者上傳的參考資料合併文字
                        (由 extract_reference_text() 產生)。
                        若有提供，AI 會被要求「依據這份真實資料」補充細節，
                        並被明確禁止編造資料中沒有的內容，藉此提升講稿的真實性。
    """
    slide_content = slide_info["title"]
    if slide_info["body_text"]:
        slide_content += "\n" + slide_info["body_text"]

    reference_block = ""
    if reference_text:
        reference_block = f"""
以下是與本課程主題相關的參考資料，請優先根據這些真實資料來補充正確的細節、數字、
法規條文、案例或專有名詞。絕對不可以編造參考資料中沒有出現的內容；
如果投影片內容與參考資料都沒有提到的細節，就不要憑空補充。

--- 參考資料開始 ---
{reference_text}
--- 參考資料結束 ---
"""

    prompt = f"""你是一位企業教育訓練講師。
請依照以下投影片內容，撰寫約30~60秒的口語旁白講稿。
{reference_block}
要求：
- 不要逐字朗讀投影片文字，要用自然口語表達，像是真人講師在課堂上說話
- 保留法規名稱、專有名詞的正確用字，不可竄改
- 使用繁體中文
- 只輸出講稿本身文字，不要加任何標題、說明或引號

投影片內容：
{slide_content}
"""
    try:
        return _call_gemini(prompt, api_key, model_name)
    except Exception as e:
        log(f"❌ 第 {slide_info['slide_index']} 頁 AI 講稿生成失敗：{e}", "error")
        return ""


def polish_script_with_ai(
    original_text: str,
    api_key: str,
    reference_text: str = "",
    model_name: str = "gemini-1.5-flash",
) -> str:
    """
    模式四：請 AI 將使用者提供的講稿修飾得更口語自然，但不改變原意。

    參數:
        reference_text: 選填。若有提供參考資料，AI 在修飾語氣的同時，
                        也可以核對/補充講稿中提到的細節是否與參考資料一致，
                        但同樣不可編造參考資料中沒有的內容。
    """
    reference_block = ""
    if reference_text:
        reference_block = f"""
以下是與本課程主題相關的參考資料，修飾講稿時可以參考這些資料確認用詞正確性，
若原始講稿有可以用參考資料補充得更精確的地方（例如法規全名、數字），可以順手修正，
但絕對不可以編造參考資料中沒有出現的內容：

--- 參考資料開始 ---
{reference_text}
--- 參考資料結束 ---
"""

    prompt = f"""你是一位教育訓練老師的講稿修飾助手。
請將以下講稿修飾成自然口語，適合教育訓練老師實際講課使用。
{reference_block}
規則：
- 絕對不要改變原意，不要增加或刪減重要資訊（例如法規名稱、數字、條文）
- 只是讓語氣更自然、更口語化，去除生硬的書面用語
- 使用繁體中文
- 只輸出修飾後的講稿文字，不要加任何說明或標題

原始講稿：
{original_text}
"""
    try:
        return _call_gemini(prompt, api_key, model_name)
    except Exception as e:
        log(f"❌ AI 講稿修飾失敗：{e}", "error")
        return original_text  # 修飾失敗時，保留原始講稿，避免內容遺失


# ------------------------------------------------------------
# 統一入口：依模式產生所有頁面的講稿
# ------------------------------------------------------------
def generate_all_scripts(
    mode: str,
    slides_text: list,
    uploaded_script_path: str = None,
    api_key: str = None,
    reference_paths: list = None,
    progress_callback=None,
) -> list:
    """
    依照選擇的旁白模式，產生「每一頁」的講稿清單。

    參數:
        mode: "own" | "ai_auto" | "notes" | "ai_polish"
        slides_text: ppt_reader.read_ppt_slides() 的回傳結果
        uploaded_script_path: 模式一/模式四需要，使用者上傳的 docx 或 txt 路徑
        api_key: 模式二/模式四需要，Gemini API Key
        reference_paths: 選填，模式二/模式四可用。使用者上傳的「參考資料」檔案路徑清單
                         (PDF/docx/txt)，AI 生成或修飾講稿時會依據這些真實資料補充細節，
                         提升講稿內容的真實性、降低憑空杜撰的風險。
        progress_callback: 選填，function(fraction: float) 用來回報進度給 Gradio 進度條

    回傳:
        list[str]，長度 = 投影片頁數
    """
    slide_count = len(slides_text)

    if mode == "notes":
        return get_notes_scripts(slides_text)

    if mode == "own":
        if not uploaded_script_path:
            raise ValueError("模式一（使用自己的講稿）需要先上傳 docx 或 txt 檔案")
        return _read_uploaded_script(uploaded_script_path, slide_count)

    if mode == "ai_auto":
        if not api_key:
            raise ValueError("模式二（AI 自動生成講稿）需要先輸入 Gemini API Key")

        reference_text = extract_reference_text(reference_paths) if reference_paths else ""
        if reference_text:
            log("📚 已讀入參考資料，AI 生成講稿時會依據這些資料補充真實細節")

        scripts = []
        for i, s in enumerate(slides_text):
            log(f"AI 生成講稿中：第 {s['slide_index']}/{slide_count} 頁")
            scripts.append(generate_script_with_ai(s, api_key, reference_text=reference_text))
            if progress_callback:
                progress_callback((i + 1) / slide_count)
        log("✅ AI 講稿自動生成完成")
        return scripts

    if mode == "ai_polish":
        if not uploaded_script_path:
            raise ValueError("模式四（AI 修飾講稿）需要先上傳您自己的 docx 或 txt 講稿")
        if not api_key:
            raise ValueError("模式四（AI 修飾講稿）需要先輸入 Gemini API Key")

        reference_text = extract_reference_text(reference_paths) if reference_paths else ""
        if reference_text:
            log("📚 已讀入參考資料，AI 修飾講稿時會依據這些資料核對真實細節")

        raw_scripts = _read_uploaded_script(uploaded_script_path, slide_count)
        polished = []
        for i, text in enumerate(raw_scripts):
            log(f"AI 修飾講稿中：第 {i + 1}/{slide_count} 頁")
            if text.strip():
                polished.append(polish_script_with_ai(text, api_key, reference_text=reference_text))
            else:
                polished.append("")
            if progress_callback:
                progress_callback((i + 1) / slide_count)
        log("✅ AI 講稿修飾完成")
        return polished

    raise ValueError(f"未知的旁白模式：{mode}")


def _read_uploaded_script(uploaded_script_path: str, slide_count: int) -> list:
    """依副檔名判斷要用 docx 或 txt 的方式讀取使用者上傳的講稿。"""
    ext = os.path.splitext(uploaded_script_path)[1].lower()
    if ext == ".docx":
        return read_script_from_docx(uploaded_script_path, slide_count)
    elif ext == ".txt":
        return read_script_from_txt(uploaded_script_path, slide_count)
    else:
        raise ValueError(f"不支援的講稿檔案格式：{ext}（僅支援 .docx 或 .txt）")


# ------------------------------------------------------------
# 儲存講稿成 Word 檔（給使用者下載，也方便配音前最後校對）
# ------------------------------------------------------------
def save_scripts_to_docx(slides_text: list, scripts: list, output_path: str):
    """
    將「每頁講稿」整理成一份 Word 文件，存到 output_path。
    對應【輸出】需求中的 講稿.docx。
    """
    doc = Document()
    doc.add_heading("課程講稿", level=1)

    for s, script in zip(slides_text, scripts):
        doc.add_heading(f"第 {s['slide_index']} 頁：{s['title'] or ''}", level=2)
        doc.add_paragraph(script.strip() if script and script.strip() else "（尚無講稿內容）")

    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    doc.save(output_path)
    log(f"✅ 講稿已儲存為 Word 檔：{output_path}")
    return output_path


# ------------------------------------------------------------
# 自我測試區塊
# ------------------------------------------------------------
if __name__ == "__main__":
    """
    測試模式一（自己的講稿）與模式三（Speaker Notes），
    這兩種模式不需要呼叫外部 AI API，可以直接離線測試。
    """
    from ppt_reader import load_presentation
    from pptx import Presentation

    test_dir = "/home/claude/PPT2Course_AI"
    test_ppt_path = os.path.join(test_dir, "PPT", "_test_script_gen.pptx")
    test_image_dir = os.path.join(test_dir, "Images", "_test_script_gen")

    # 建立 3 頁測試 PPT（含備忘稿）
    prs = Presentation()
    layout = prs.slide_layouts[1]
    sample_slides = [
        ("課程介紹", "本課程將說明公司資訊安全政策的重要性。", "各位同仁大家好，歡迎參加本次教育訓練課程。"),
        ("資安法規概述", "個人資料保護法\n資通安全管理法", "這一頁我們來看看重要的法規。"),
        ("結語與Q&A", "感謝大家的參與，如有問題歡迎提出。", "課程到此結束，謝謝大家。"),
    ]
    for title, body, notes in sample_slides:
        slide = prs.slides.add_slide(layout)
        slide.shapes.title.text = title
        slide.placeholders[1].text = body
        slide.notes_slide.notes_text_frame.text = notes
    prs.save(test_ppt_path)

    result = load_presentation(test_ppt_path, test_image_dir)
    slides_text = result["slides_text"]

    print("\n===== 測試模式三：Speaker Notes =====")
    notes_scripts = generate_all_scripts("notes", slides_text)
    for s in notes_scripts:
        print(repr(s))

    # 建立測試用講稿 docx（模式一）
    test_script_docx = os.path.join(test_dir, "Script", "_test_user_script.docx")
    doc = Document()
    doc.add_paragraph("第1頁")
    doc.add_paragraph("大家好，這是第一頁的自訂講稿內容，用來測試模式一的頁碼對應功能。")
    doc.add_paragraph("第2頁")
    doc.add_paragraph("這是第二頁的講稿，介紹個資法與資安法。")
    doc.add_paragraph("第3頁")
    doc.add_paragraph("最後一頁，感謝聆聽，歡迎提問。")
    doc.save(test_script_docx)

    print("\n===== 測試模式一：使用者 docx 講稿 =====")
    own_scripts = generate_all_scripts("own", slides_text, uploaded_script_path=test_script_docx)
    for s in own_scripts:
        print(repr(s))

    print("\n===== 測試講稿預覽格式化 / 反解析 =====")
    display_text = format_scripts_for_display(slides_text, own_scripts)
    print(display_text)
    reparsed = parse_scripts_from_display(display_text, len(slides_text))
    assert reparsed == own_scripts, "反解析結果與原始講稿不一致！"
    print("\n✅ 格式化與反解析一致性測試通過")

    print("\n===== 測試儲存為 Word =====")
    output_docx = os.path.join(test_dir, "Output", "_test_講稿.docx")
    save_scripts_to_docx(slides_text, own_scripts, output_docx)
    print("存在:", os.path.exists(output_docx))

    # --- 測試參考資料讀取功能 (PDF / docx / txt) ---
    print("\n===== 測試參考資料讀取 (extract_reference_text) =====")
    from reportlab.pdfgen import canvas as rl_canvas

    ref_pdf_path = os.path.join(test_dir, "Assets", "_test_ref.pdf")
    c = rl_canvas.Canvas(ref_pdf_path)
    c.drawString(100, 800, "Reference PDF Test Content 12345")
    c.save()

    ref_docx_path = os.path.join(test_dir, "Assets", "_test_ref.docx")
    ref_doc = Document()
    ref_doc.add_paragraph("這是參考資料docx測試內容，關鍵字：資安法規測試999")
    ref_doc.save(ref_docx_path)

    ref_txt_path = os.path.join(test_dir, "Assets", "_test_ref.txt")
    with open(ref_txt_path, "w", encoding="utf-8") as f:
        f.write("這是txt參考資料測試內容，關鍵字：個資保護測試888")

    combined = extract_reference_text([ref_pdf_path, ref_docx_path, ref_txt_path])
    print(combined)

    assert "12345" in combined, "PDF 內容擷取失敗"
    assert "999" in combined, "docx 內容擷取失敗"
    assert "888" in combined, "txt 內容擷取失敗"
    print("\n✅ 參考資料讀取（PDF/docx/txt）測試通過")

    # 清理參考資料測試檔案
    for p in [ref_pdf_path, ref_docx_path, ref_txt_path]:
        os.remove(p)


Writing script_generator.py


In [7]:
%%writefile app.py
# -*- coding: utf-8 -*-
"""
app.py
======
PPT2Course AI 系統的主程式進入點，使用 Gradio 建立網頁介面。

【目前進度：Step 2】
Step 1：上傳 PPT → 解析內容 → 預覽（頁數 / 圖片 / 文字 / 備忘稿）
Step 2：選擇四種旁白模式之一 → 產生每頁講稿 → 可直接編輯 → 下載成 Word

之後的步驟會逐步加入：
    Step 3: AI 配音 (tts.py)
    Step 4: 字幕產生 (subtitle.py)
    Step 5: 影片合成 (video.py)
    Step 6: Logo / 背景音樂 / 片頭片尾 / 完整輸出與打包
"""

import os
import gradio as gr

from utils import (
    ensure_all_dirs,
    check_environment,
    make_session_dir,
    safe_filename,
    log,
    get_log_text,
    clear_log,
    DIR_PPT,
    DIR_IMAGES,
    DIR_SCRIPT,
    DIR_OUTPUT,
)
from ppt_reader import load_presentation
from script_generator import (
    generate_all_scripts,
    format_scripts_for_display,
    parse_scripts_from_display,
    save_scripts_to_docx,
)


# ------------------------------------------------------------
# 初始化：確保資料夾存在、檢查環境
# ------------------------------------------------------------
ensure_all_dirs()
ENV_STATUS = check_environment()

# 旁白模式：畫面上顯示的中文選項 <-> 程式內部使用的英文代碼
MODE_LABEL_TO_CODE = {
    "① 使用自己的講稿 (上傳 docx / txt)": "own",
    "② AI 自動生成講稿 (需要 Gemini API Key)": "ai_auto",
    "③ 使用 PowerPoint 備忘稿 (Speaker Notes)": "notes",
    "④ AI 修飾自己的講稿 (上傳講稿 + Gemini API Key)": "ai_polish",
}


# ------------------------------------------------------------
# Step 1 的功能：處理使用者上傳的 PPT
# ------------------------------------------------------------
def handle_ppt_upload(ppt_file):
    """
    當使用者上傳 PPT 並按下「解析 PPT」按鈕時執行。

    回傳 (依序對應到 Gradio 介面上的元件):
        1. summary_output: 文字說明，顯示共幾頁
        2. gallery_output: 每頁投影片圖片
        3. preview_output: 文字/備忘稿預覽 (Markdown)
        4. log_output: 執行紀錄
        5. slides_state: 存到 gr.State，供 Step2 講稿生成使用
    """
    clear_log()

    if ppt_file is None:
        return "請先上傳 PPT 檔案", [], "", get_log_text(), None

    original_path = ppt_file.name if hasattr(ppt_file, "name") else ppt_file
    filename = safe_filename(os.path.basename(original_path))

    saved_ppt_path = os.path.join(DIR_PPT, filename)
    with open(original_path, "rb") as src, open(saved_ppt_path, "wb") as dst:
        dst.write(src.read())

    log(f"已收到上傳檔案：{filename}")

    session_image_dir = make_session_dir(DIR_IMAGES)

    try:
        result = load_presentation(saved_ppt_path, session_image_dir)
    except Exception as e:
        log(f"❌ 解析失敗：{e}", "error")
        return f"解析失敗：{e}", [], "", get_log_text(), None

    slide_count = result["slide_count"]
    images = result["slides_images"]
    slides_text = result["slides_text"]

    preview_lines = []
    for s in slides_text:
        preview_lines.append(f"### 第 {s['slide_index']} 頁：{s['title'] or '(無標題)'}")
        if s["body_text"]:
            preview_lines.append(f"**內文：** {s['body_text']}")
        if s["notes"]:
            preview_lines.append(f"**備忘稿：** {s['notes']}")
        preview_lines.append("---")
    preview_text = "\n\n".join(preview_lines)

    summary_text = f"✅ 解析完成，共 {slide_count} 頁投影片。可以繼續下方「產生講稿」步驟了。"

    # slides_text 存進 gr.State，供後面產生講稿使用（不用重新解析 PPT）
    return summary_text, images, preview_text, get_log_text(), slides_text


# ------------------------------------------------------------
# Step 2 的功能：依模式產生講稿
# ------------------------------------------------------------
def toggle_mode_inputs(mode_label):
    """
    根據使用者選擇的旁白模式，決定要顯示/隱藏哪些輸入欄位：
        - 模式一 (own)      ：顯示 講稿上傳
        - 模式二 (ai_auto)  ：顯示 API Key + 參考資料上傳
        - 模式三 (notes)    ：都不顯示（直接用備忘稿）
        - 模式四 (ai_polish)：顯示 講稿上傳 + API Key + 參考資料上傳
    """
    mode = MODE_LABEL_TO_CODE.get(mode_label, "notes")
    show_script_upload = mode in ("own", "ai_polish")
    show_api_key = mode in ("ai_auto", "ai_polish")
    show_reference = mode in ("ai_auto", "ai_polish")
    return (
        gr.update(visible=show_script_upload),
        gr.update(visible=show_api_key),
        gr.update(visible=show_reference),
    )


def handle_generate_script(mode_label, slides_state, script_file, api_key, reference_files):
    """
    當使用者按下「生成講稿」按鈕時執行。

    參數:
        mode_label: 使用者在 Radio 選單選擇的中文標籤
        slides_state: Step1 存下來的投影片文字資料 (list[dict])
        script_file: 使用者上傳的講稿檔案（模式一/四才需要）
        api_key: Gemini API Key（模式二/四才需要）
        reference_files: 使用者上傳的參考資料檔案清單（模式二/四可選填）。
                         有上傳的話，AI 生成/修飾講稿時會依據這些真實資料補充細節。

    回傳:
        1. script_preview: 可編輯的講稿全文（含頁碼標記）
        2. log_output: 執行紀錄
    """
    clear_log()

    if not slides_state:
        return "請先完成上方「Step 1：解析 PPT」，再產生講稿。", get_log_text()

    mode = MODE_LABEL_TO_CODE.get(mode_label)
    script_path = None
    if script_file is not None:
        script_path = script_file.name if hasattr(script_file, "name") else script_file

    reference_paths = []
    if reference_files:
        # gr.File(file_count="multiple") 回傳的是一個 list，每個元素有 .name 屬性
        for f in reference_files:
            reference_paths.append(f.name if hasattr(f, "name") else f)

    try:
        scripts = generate_all_scripts(
            mode=mode,
            slides_text=slides_state,
            uploaded_script_path=script_path,
            api_key=api_key if api_key else None,
            reference_paths=reference_paths,
        )
    except Exception as e:
        log(f"❌ 講稿生成失敗：{e}", "error")
        return f"❌ 講稿生成失敗：{e}", get_log_text()

    display_text = format_scripts_for_display(slides_state, scripts)
    return display_text, get_log_text()


def handle_save_script(script_preview_text, slides_state):
    """
    當使用者按下「儲存講稿為 Word」按鈕時執行。
    會把目前編輯框中的文字（使用者可能已手動修改過）重新解析回每頁講稿，
    再輸出成 Output/講稿.docx，讓使用者下載。
    """
    clear_log()

    if not slides_state:
        return None, "請先完成 Step 1 與 Step 2，再儲存講稿。"

    if not script_preview_text or not script_preview_text.strip():
        return None, "講稿內容是空的，請先產生或輸入講稿。"

    scripts = parse_scripts_from_display(script_preview_text, len(slides_state))
    output_path = os.path.join(DIR_OUTPUT, "講稿.docx")

    try:
        save_scripts_to_docx(slides_state, scripts, output_path)
    except Exception as e:
        log(f"❌ 儲存講稿失敗：{e}", "error")
        return None, get_log_text()

    return output_path, get_log_text()


# ------------------------------------------------------------
# 建立 Gradio 介面
# ------------------------------------------------------------
def build_ui():
    """
    建立並回傳 Gradio Blocks 介面物件。
    """
    with gr.Blocks(title="PPT2Course AI") as demo:
        gr.Markdown(
            """
            # 🎓 PPT2Course AI
            ### 一鍵將 PowerPoint 轉換成完整線上課程影片
            **目前進度：Step 2 - 講稿生成（四種旁白模式）**
            """
        )

        if not ENV_STATUS["all_ok"]:
            missing = []
            if not ENV_STATUS["libreoffice"]:
                missing.append("LibreOffice")
            if not ENV_STATUS["ffmpeg"]:
                missing.append("FFmpeg")
            gr.Markdown(
                f"⚠️ **注意：系統缺少以下工具，部分功能可能無法使用：{', '.join(missing)}**"
            )

        # gr.State：在使用者操作過程中，暫存 Step1 解析出來的投影片資料，
        # 讓 Step2 可以直接使用，不需要重新解析 PPT。
        slides_state = gr.State(value=None)

        # ---------------- Step 1：上傳 PPT ----------------
        gr.Markdown("## 1️⃣ 上傳並解析 PPT")
        with gr.Row():
            with gr.Column(scale=1):
                ppt_input = gr.File(
                    label="上傳 PowerPoint 檔案 (.pptx)",
                    file_types=[".pptx", ".ppt"],
                )
                parse_button = gr.Button("🔍 解析 PPT", variant="primary")
                summary_output = gr.Textbox(label="解析結果摘要", interactive=False)

            with gr.Column(scale=2):
                gallery_output = gr.Gallery(
                    label="投影片預覽", columns=4, height="auto"
                )

        with gr.Accordion("📄 投影片文字內容預覽", open=False):
            preview_output = gr.Markdown()

        # ---------------- Step 2：產生講稿 ----------------
        gr.Markdown("## 2️⃣ 產生講稿（請選擇一種旁白模式）")

        mode_radio = gr.Radio(
            choices=list(MODE_LABEL_TO_CODE.keys()),
            value="③ 使用 PowerPoint 備忘稿 (Speaker Notes)",
            label="旁白模式",
        )

        with gr.Row():
            script_upload = gr.File(
                label="📎 上傳您的講稿檔案 (.docx 或 .txt)｜模式①④ 需要",
                file_types=[".docx", ".txt"],
                visible=False,
            )
            api_key_input = gr.Textbox(
                label="🔑 Gemini API Key｜模式②④ 需要",
                type="password",
                placeholder="在此貼上您的 Gemini API Key",
                visible=False,
            )

        reference_upload = gr.File(
            label="📚 上傳參考資料（選填）｜可上傳多份 PDF / docx / txt，"
            "AI 會依據這些真實資料補充講稿細節，避免內容失真",
            file_types=[".pdf", ".docx", ".txt"],
            file_count="multiple",
            visible=False,
        )

        gr.Markdown(
            "💡 還沒有 Gemini API Key？可以到 "
            "[Google AI Studio](https://aistudio.google.com/app/apikey) "
            "免費申請一組。\n\n"
            "💡 想讓 AI 產出的講稿更貼近真實內容嗎？可以把相關的法規全文、公司政策文件、"
            "報告等資料放進「參考資料」上傳（也可以先丟進 NotebookLM 整理消化過，"
            "再把整理好的內容存成 txt/docx 上傳）。"
        )

        generate_script_button = gr.Button("✍️ 生成講稿", variant="primary")

        script_preview = gr.Textbox(
            label="講稿預覽 / 編輯（可直接修改文字，修改後請按下方「儲存」）",
            lines=18,
        )

        save_script_button = gr.Button("💾 儲存講稿為 Word 檔")
        script_download = gr.File(label="下載 講稿.docx")

        # ---------------- 共用：執行紀錄 ----------------
        with gr.Accordion("📋 即時執行紀錄 (Log)", open=False):
            log_output = gr.Textbox(label="Log", lines=10, interactive=False)

        # ---------------- 事件綁定 ----------------
        parse_button.click(
            fn=handle_ppt_upload,
            inputs=[ppt_input],
            outputs=[summary_output, gallery_output, preview_output, log_output, slides_state],
        )

        mode_radio.change(
            fn=toggle_mode_inputs,
            inputs=[mode_radio],
            outputs=[script_upload, api_key_input, reference_upload],
        )

        generate_script_button.click(
            fn=handle_generate_script,
            inputs=[mode_radio, slides_state, script_upload, api_key_input, reference_upload],
            outputs=[script_preview, log_output],
        )

        save_script_button.click(
            fn=handle_save_script,
            inputs=[script_preview, slides_state],
            outputs=[script_download, log_output],
        )

    return demo


# ------------------------------------------------------------
# 主程式進入點
# ------------------------------------------------------------
if __name__ == "__main__":
    demo = build_ui()
    demo.launch(share=True)  # share=True 會產生一組公開網址，方便在 Colab 使用


Writing app.py


In [8]:
import os
for d in ['PPT', 'Script', 'Audio', 'Images', 'Output', 'Assets']:
    os.makedirs(d, exist_ok=True)
print('✅ 資料夾建立完成')


✅ 資料夾建立完成


In [10]:
import sys
!{sys.executable} -m pip install python-pptx pdf2image python-docx
%run app.py

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://42924a89edf35daa26.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
